# Amsterdam: a case built in Python

`run_file` ([the first example](delft.ipynb)) does three things before it runs: it builds a network, a timetable and
the travellers from the case's files. This example does the same steps one by one in Python on a larger case,
Amsterdam's morning (198 252 travellers), and then asks a question of the model: what changes if every bike rides at
18 km/h instead of 15 km/h?

It needs `pip install openmobisim pyarrow`. Everything below was run as shown, on a laptop (Apple M2 Pro, 12 cores).
The travellers are synthetic and the choice coefficients are the defaults, not fitted to Amsterdam: the result shows
how the model responds, and is not a forecast.

In [1]:
import urllib.request, zipfile

urllib.request.urlretrieve("https://zenodo.org/records/24585967/files/amsterdam.zip", "amsterdam.zip")
zipfile.ZipFile("amsterdam.zip").extractall()

## The settings

The case's `run.toml` holds the window, the classes of travellers and every parameter. It is plain TOML.

In [2]:
import tomllib

with open("amsterdam/run.toml", "rb") as f:
    settings = tomllib.load(f)
window = (settings["window_start_s"], settings["window_end_s"])   # seconds after midnight: 07:30 to 09:30
classes = settings["demand"]["classes"]
print(window, classes)
print({name: value for name, value in settings["parameters"].items() if name != "coefficients"})

(27000, 34200) ['bike_only', 'car_and_bike', 'car_only', 'from_outside_car', 'from_outside_transit', 'goods', 'no_vehicle']
{'capacity_per_lane': 1800.0, 'bike_speed': 15.0, 'walk_speed': 4.8, 'stop_walk_max': 1500.0, 'transfer_walk_max': 400.0, 'boarding_slack': 60, 'max_rides': 4, 'nearby_stops': 3, 'max_car_routes': 6, 'max_transit_journeys': 10, 'choice': 'logit', 'nest_scale': 0.5, 'step_exponent': 0.75, 'iteration_cap': 50, 'iteration_floor': 0, 'link_bin': 300, 'seed': 0}


Every parameter in the file is at its default ([the parameters table](../parameters.md)), and so are the choice
coefficients of every class, so the calls below give none of them: a keyword left out takes its default.

## The network, the timetable, the travellers

A table is given in memory (a dict of lists, a pandas or polars frame, a pyarrow table) or, as here, as the path of a
Parquet file. The timetable is read on the network, and the travellers on both: each trip end is linked to the
network per mode and to the stops within walking distance.

In [3]:
import time
import openmobisim as oms

def build(**network_parameters):
    network = oms.Network.from_tables("amsterdam/nodes.parquet", "amsterdam/links.parquet", **network_parameters)
    transit = oms.Transit.from_gtfs("amsterdam/gtfs.zip", network)
    demand = oms.Demand.from_table("amsterdam/trips.parquet", classes, network, transit=transit)
    return network, transit, demand

start = time.perf_counter()
network, transit, demand = build()
print(f"built in {time.perf_counter() - start:.1f} s")
print("repairs:", network.repairs, transit.repairs, demand.repairs)

built in 3.1 s
repairs: {} {} {}


A reader sets right or leaves out what it cannot read instead of refusing the whole table (a link naming a node the
nodes table does not hold is left out, a trip of a class not given is dropped), says so in a warning and counts it in
`repairs`; these files need none.

## The run

In [4]:
start = time.perf_counter()
base = oms.run(network, demand, transit, window=window, out="amsterdam-run")
print(f"run time {time.perf_counter() - start:.1f} s; ended by the stopping rule: {base.stopped_by_rule}")

run time 50.0 s; ended by the stopping rule: True


In [5]:
import pyarrow.parquet as pq

def by_mode(trips_file):
    """Per mode: the travellers who took it, their share (%) and their mean door-to-door time (min)."""
    trips = pq.read_table(trips_file).to_pylist()
    rows = {}
    for mode in ("car", "bike", "walk", "transit"):
        times = [t["travel_time_s"] for t in trips if t["mode"] == mode and t["finished"]]
        count = sum(t["mode"] == mode for t in trips)
        rows[mode] = (count, 100 * count / len(trips), sum(times) / len(times) / 60)
    return rows

def show(rows):
    print("mode     travellers  share (%)  mean door-to-door time (min)")
    for mode, (count, share, minutes) in rows.items():
        print(f"{mode:8} {count:10} {share:10.1f} {minutes:10.1f}")

show(by_mode(base.trips))
print("unfinished at 09:30:", pq.read_table(base.trips, columns=["finished"])["finished"].to_pylist().count(False))

mode     travellers  share (%)  mean door-to-door time (min)
car           93729       47.3       11.7
bike          41725       21.0       22.7
walk           6254        3.2       26.7
transit       56544       28.5       36.1
unfinished at 09:30: 0


These are the results `oms.run_file("amsterdam", out=...)` gives: the same inputs and settings give the same trips,
links and routes tables.

## A second run: bikes at 18 km/h

`bike_speed` is the one speed of every bike (15 km/h by default). It belongs to the network, which holds each link's
time per mode, so the network is built again with the new value, and the timetable and the travellers on that
network. Nothing else changes: the same travellers, the same choice model, the same seed.

In [6]:
start = time.perf_counter()
network_18, transit_18, demand_18 = build(bike_speed=18)
faster = oms.run(network_18, demand_18, transit_18, window=window, out="amsterdam-run-bikes-18")
print(f"built and run in {time.perf_counter() - start:.1f} s; ended by the stopping rule: {faster.stopped_by_rule}")
show(by_mode(faster.trips))
print("unfinished at 09:30:", pq.read_table(faster.trips, columns=["finished"])["finished"].to_pylist().count(False))

built and run in 57.5 s; ended by the stopping rule: True
mode     travellers  share (%)  mean door-to-door time (min)
car           90916       45.9       11.6
bike          45294       22.8       19.5
walk           6217        3.1       26.7
transit       55825       28.2       36.1
unfinished at 09:30: 1


One trip is unfinished in the second run: a traveller who leaves at 08:26 and took transit in the first run (48 min)
now takes the bike for a ride of about 65 min, and is still riding when the window ends at 09:30. A trip under way at
the end of the window is counted as unfinished and has no travel time, so it is not in the mean.

In [7]:
before, after = by_mode(base.trips), by_mode(faster.trips)
print("mode     travellers at 15 km/h  at 18 km/h  change (travellers)  mean time at 15 km/h (min)  at 18 km/h (min)")
for mode in before:
    print(f"{mode:8} {before[mode][0]:21} {after[mode][0]:11} {after[mode][0] - before[mode][0]:+20} {before[mode][2]:27.1f} {after[mode][2]:17.1f}")

mode     travellers at 15 km/h  at 18 km/h  change (travellers)  mean time at 15 km/h (min)  at 18 km/h (min)
car                      93729       90916                -2813                        11.7              11.6
bike                     41725       45294                +3569                        22.7              19.5
walk                      6254        6217                  -37                        26.7              26.7
transit                  56544       55825                 -719                        36.1              36.1


Travellers moved to the bike from each other mode. The mean bike trip fell to 0.86 of what it was, not to
15/18 = 0.83. The links table gives the distance travelled on the network in both runs, and so the distance per
cyclist:

In [8]:
network_links = pq.read_table("amsterdam/links.parquet", columns=["link_id", "length_m"])
length_m = dict(zip(network_links["link_id"].to_pylist(), network_links["length_m"].to_pylist()))

def km(run, column):
    table = pq.read_table(run.links, columns=["link_id", column])
    return sum(n * length_m[l] for l, n in zip(table["link_id"].to_pylist(), table[column].to_pylist())) / 1000

for mode, column in (("car", "cars"), ("bike", "bikes"), ("walk", "walkers")):
    print(f"{mode:5} {km(base, column):9.0f} km at 15 km/h {km(faster, column):9.0f} km at 18 km/h")
print(f"per cyclist: {km(base, 'bikes') / before['bike'][0]:.2f} km at 15 km/h, {km(faster, 'bikes') / after['bike'][0]:.2f} km at 18 km/h")

car      991629 km at 15 km/h    965770 km at 18 km/h


bike     222087 km at 15 km/h    245434 km at 18 km/h


walk      12822 km at 15 km/h     12764 km at 18 km/h


per cyclist: 5.32 km at 15 km/h, 5.42 km at 18 km/h


The cyclists of the second run ride a little farther on average, and the walk between a door and the network is
part of every trip and stays at walking speed: two reasons why the mean time falls by less than the speed rises.

## The pages of both runs

Each run writes its own pages; the title distinguishes them.

In [9]:
import os

base.map("amsterdam-map.html", title="Amsterdam, bikes at 15 km/h")
faster.map("amsterdam-map-bikes-18.html", title="Amsterdam, bikes at 18 km/h")
base.summary("amsterdam-summary.html", title="Amsterdam, bikes at 15 km/h")
faster.summary("amsterdam-summary-bikes-18.html", title="Amsterdam, bikes at 18 km/h")
for page in ("amsterdam-map.html", "amsterdam-map-bikes-18.html", "amsterdam-summary.html", "amsterdam-summary-bikes-18.html"):
    print(f"{page}: {os.path.getsize(page) / 1e6:.1f} MB")

amsterdam-map.html: 16.8 MB
amsterdam-map-bikes-18.html: 16.8 MB
amsterdam-summary.html: 12.4 MB
amsterdam-summary-bikes-18.html: 12.4 MB


![The map page of the second run with its cyclists layer on: Amsterdam's cycle network, each link's width the cyclists who rode it over the run](../_static/figures/amsterdam_map_bikes.png)

The map of the second run with the layer of cyclists switched on (the network is from OpenStreetMap,
© OpenStreetMap contributors).

Each run's record, `run_record.toml`, holds the settings it ran with, `bike_speed` among them, so the folder alone
says which run it is, and its first part copied into a `run.toml` runs it again.

In [10]:
for run in (base, faster):
    record = open(run.record).read()
    print(next(line for line in record.splitlines() if line.startswith("bike_speed")))

bike_speed = 15.0  # km/h
bike_speed = 18.0  # km/h
